# Per-image сравнение описаний из выбранных экспериментов

Структура: изображение → таблица с описаниями из всех выбранных experiments.

Гибкий ноутбук: меняешь только список `EXPS` в первой ячейке.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown

# === SETUP ===
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB = ROOT / 'data' / 'eval' / 'experiments.db'
assert DB.exists(), f'DB not found at {DB}'
conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row

# === ВЫБОР ЭКСПЕРИМЕНТОВ ДЛЯ СРАВНЕНИЯ ===
# Поменяй список по необходимости. По умолчанию — последние / самые релевантные.
EXPS = [
    'E00_postproc_display_n60',           # template baseline + legacy taxonomy, no Vikhr
    'E08_artcap_v2soft_n60',              # ArtCap-LoRA + Vikhr v2 + legacy taxonomy
    'E06b_archival_v2_n60',               # Vikhr v2 + archival_v2 taxonomy (NEW)
]

# Короткие labels для отображения
SHORT_LABELS = {
    'E00_postproc_display_n60':       'E00 template (legacy)',
    'E08_artcap_v2soft_n60':          'E08 ArtCap+Vikhr (legacy)',
    'E06b_archival_v2_n60':           'E06b Vikhr (archival_v2)',
    # n=14 semtest experiments — uncomment if comparing semtest:
    'E12_v1_semtest':   'E12 v1 (raw)',
    'E12_v2_semtest':   'E12 v2 (few-shot)',
    'E12_v3_semtest':   'E12 v3 (strict)',
    'E00_semtest':      'E00 template (legacy)',
    'E05d_semtest':     'E05d smart template',
    'E08_artcap_epoch2_semtest':    'E08 ArtCap-v1 (aggressive)',
    'E08_artcap_v2soft_semtest':    'E08 ArtCap-v2_soft',
}

# Filter EXPS to only those actually in DB
in_db = {r['exp_id'] for r in conn.execute('SELECT exp_id FROM experiments').fetchall()}
EXPS = [e for e in EXPS if e in in_db]
print(f'Comparing {len(EXPS)} experiments:')
for e in EXPS:
    print(f'  - {e}')

In [ ]:
# Load all per_item rows for selected experiments
df = pd.read_sql(
    f"""SELECT e.exp_id, p.image_path, p.source, p.reference_ru,
               p.caption_en, p.caption_ru, p.archive_ru,
               p.sds_value, p.retrieval_rank_t2i,
               (SELECT value FROM metrics m
                WHERE m.experiment_id=e.id AND m.metric_name='CLIPScore_RU_archive_ru' AND m.source='all') AS clipscore_overall
        FROM per_item p
        JOIN experiments e ON e.id = p.experiment_id
        WHERE e.exp_id IN ({','.join('?'*len(EXPS))})
        ORDER BY p.image_path, e.exp_id""",
    conn, params=EXPS,
)
df['image'] = df['image_path'].apply(lambda p: Path(p).name)

# Show which images are present in ALL selected experiments
img_counts = df.groupby('image')['exp_id'].nunique()
common_imgs = img_counts[img_counts == len(EXPS)].index.tolist()
print(f'Total unique images: {df["image"].nunique()}')
print(f'Common across ALL {len(EXPS)} exps: {len(common_imgs)}')

# Order: RGB (postcard_N) first by index, then NYPL hashes
def img_sort_key(name):
    if name.startswith('postcard_'):
        try: return (0, int(name.split('_')[1].split('.')[0]))
        except: return (0, 0)
    return (1, name)

image_order = sorted(common_imgs, key=img_sort_key)
print(f'Will display: {len(image_order)} images')

## Per-image сравнение

In [ ]:
for image_name in image_order:
    sub = df[df['image'] == image_name].copy()
    if sub.empty:
        continue
    
    image_path = sub.iloc[0]['image_path']
    source = sub.iloc[0]['source']
    ref = sub.iloc[0]['reference_ru'] or '(нет reference)'
    abs_path = ROOT / image_path
    
    display(Markdown(f'### {image_name}  &nbsp; *[{source}]*'))
    if abs_path.exists():
        display(Image(filename=str(abs_path), width=320))
    display(Markdown(f'**REF:** {ref}'))
    
    # Order experiments as in EXPS list
    sub['ord'] = sub['exp_id'].apply(lambda x: EXPS.index(x) if x in EXPS else 99)
    sub = sub.sort_values('ord')
    
    rows = []
    for _, r in sub.iterrows():
        rows.append({
            'experiment': SHORT_LABELS.get(r['exp_id'], r['exp_id']),
            'caption_en': (r['caption_en'] or '')[:100],
            'archive_description': (r['archive_ru'] or '')[:200],
            'SDS': f"{r['sds_value']:.2f}" if r['sds_value'] is not None else '-',
            'rank_t2i': str(r['retrieval_rank_t2i']) if r['retrieval_rank_t2i'] is not None else '-',
        })
    display(pd.DataFrame(rows))
    display(Markdown('---'))

## Aggregate metrics across selected experiments

In [ ]:
metrics_df = pd.read_sql(
    f"""SELECT e.exp_id, m.metric_name, m.source, m.n, m.value
        FROM metrics m JOIN experiments e ON e.id = m.experiment_id
        WHERE e.exp_id IN ({','.join('?'*len(EXPS))})
          AND m.source='all'
          AND m.metric_name IN (
              'CLIPScore_EN_caption_en','CLIPScore_RU_caption_ru','CLIPScore_RU_archive_ru',
              't2i_R@1','t2i_R@5','t2i_R@10','SDS_mean','latency_mean_sec'
          )""",
    conn, params=EXPS,
)
pivot = metrics_df.pivot_table(index='metric_name', columns='exp_id', values='value', aggfunc='first')
pivot = pivot[[e for e in EXPS if e in pivot.columns]]  # preserve order
pivot.columns = [SHORT_LABELS.get(c, c) for c in pivot.columns]
pivot.style.format('{:.4f}')

## Per-axis SDS — какие оси покрыты в каждом эксперименте

In [ ]:
axes = ['type_of_material','visual_subject','artistic_style','epoch','cultural_context','mood']
axis_metrics = [f'SDS_axis_{a}' for a in axes]

sds_df = pd.read_sql(
    f"""SELECT e.exp_id, m.metric_name, m.value
        FROM metrics m JOIN experiments e ON e.id = m.experiment_id
        WHERE e.exp_id IN ({','.join('?'*len(EXPS))})
          AND m.source='all'
          AND m.metric_name IN ({','.join('?'*len(axis_metrics))})""",
    conn, params=EXPS + axis_metrics,
)
sds_pivot = sds_df.pivot_table(index='metric_name', columns='exp_id', values='value', aggfunc='first')
sds_pivot = sds_pivot[[e for e in EXPS if e in sds_pivot.columns]]
sds_pivot.columns = [SHORT_LABELS.get(c, c) for c in sds_pivot.columns]
sds_pivot.index = [i.replace('SDS_axis_','') for i in sds_pivot.index]
sds_pivot.style.format('{:.2f}').background_gradient(cmap='YlGn', axis=None)